# CD Atlas: Treg Pseudotime Trajectory (Palantir)

Computes a Palantir pseudotime trajectory for Tregs using a
pre-selected cluster 0 centroid as the root cell.

The resulting pseudotime bins (B0–B4) stratify Tregs into
Early (B0–B1) and Late (B2–B4) states used by
`08_cd_nichenet_treg.R` for NicheNet prioritization.

**Input**: `treg_strict_pseudotime.h5ad`
  (Treg + immune sender cells with UMAP and leiden clustering)

In [ ]:
import palantir
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
print("Palantir version:", palantir.__version__)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cd/scrna/output"
MERGED_DIR = f"{DATA_DIR}/merged_cd"

## 1. Load Treg + immune sender object

In [ ]:
treg = sc.read_h5ad(f"{MERGED_DIR}/treg_strict_pseudotime.h5ad")
print(treg)
print(treg.obs['leiden_0.6'].value_counts())

## 1b. Gene module scoring and Leiden UMAP (Extended Data Fig. 8A-C)

In [ ]:
# Leiden clusters UMAP (Extended Data Fig. 8A)
sc.pl.umap(treg, color='leiden_0.6', wspace=0.1, s=18, legend_loc='on data', frameon=False)

# Gene module scoring (Extended Data Fig. 8B-C)
panels = {
 'Suppressive': ['BATF','PRDM1','MAF','TNFRSF18','TNFRSF4'],
 'Tfr': ['BCL6','CXCR5','PDCD1','ICOS'],
 'Th17_like': ['RORC','CCR6','IL17A','IL17F'],
 'IFN_response': ['ISG15','IFI6','MX1','IFIT3'],
 'Cell_cycle': ['MKI67','TOP2A','STMN1'],
 'Treg_core':['FOXP3','IL2RA','CTLA4','TIGIT','ENTPD1','LRRC32','IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(treg, [g for g in genes if g in treg.var_names], score_name=name)

for name in panels.keys():
    sc.pl.umap(
        treg,
        color=name,
        cmap="viridis",
        vmin='p1', vmax='p99',
        frameon=False, s=10
    )

## 2. Diffusion maps and MAGIC imputation

In [ ]:
dm_res    = palantir.utils.run_diffusion_maps(treg, n_components=5)
ms_data   = palantir.utils.determine_multiscale_space(treg)
imputed_X = palantir.utils.run_magic_imputation(treg)

In [ ]:
palantir.plot.plot_diffusion_components(treg)
plt.show()

## 3. Select root cell: cluster 0 centroid

In [ ]:
coords = treg.obsm['X_umap']

# Root: cell closest to cluster 0 centroid
early_cells = treg.obs_names[treg.obs['leiden_0.6'] == '0']
idx         = treg.obs_names.get_indexer(early_cells)
centroid    = coords[idx].mean(axis=0)
root_cell   = early_cells[np.argmin(((coords[idx] - centroid)**2).sum(axis=1))]
print(f"Root cell: {root_cell}")

palantir.plot.highlight_cells_on_umap(treg, pd.Series(['Root'], index=[root_cell]), s=5)
plt.show()

## 4. Run Palantir pseudotime

In [ ]:
pr_res = palantir.core.run_palantir(treg, root_cell, num_waypoints=500)

palantir.plot.plot_palantir_results(treg, s=3)
plt.show()

## 4b. Alternative root cells (Extended Data Fig. 8E)

In [ ]:
treg_cp = treg.copy()
treg_cp.obs["umap1"] = treg_cp.obsm["X_umap"][:,0]
treg_cp.obs["umap2"] = treg_cp.obsm["X_umap"][:,1]

# Alternative root 1: cluster 1 centroid
early_cluster = "1"
early_cells = treg_cp.obs_names[treg_cp.obs["leiden_0.6"] == early_cluster]
coords = treg_cp.obsm["X_umap"]
idx = treg_cp.obs_names.get_indexer(early_cells)
centroid = coords[idx].mean(axis=0)
alt_root_1 = early_cells[np.argmin(((coords[idx] - centroid)**2).sum(axis=1))]

palantir.plot.highlight_cells_on_umap(treg_cp, pd.Series(['Alt Root (Clus1)'], index=[alt_root_1]), s=5)
plt.show()

pr_res_alt1 = palantir.core.run_palantir(treg_cp, alt_root_1, num_waypoints=500)
palantir.plot.plot_palantir_results(treg_cp, s=3)
plt.show()

# Alternative root 2: cluster 2 centroid
early_cluster = "2"
early_cells = treg_cp.obs_names[treg_cp.obs["leiden_0.6"] == early_cluster]
idx = treg_cp.obs_names.get_indexer(early_cells)
centroid = coords[idx].mean(axis=0)
alt_root_2 = early_cells[np.argmin(((coords[idx] - centroid)**2).sum(axis=1))]

palantir.plot.highlight_cells_on_umap(treg_cp, pd.Series(['Alt Root (Clus2)'], index=[alt_root_2]), s=5)
plt.show()

pr_res_alt2 = palantir.core.run_palantir(treg_cp, alt_root_2, num_waypoints=500)
palantir.plot.plot_palantir_results(treg_cp, s=3)
plt.show()

## 4c. DPT with PAGA (Extended Data Fig. 8F) and pathway / TF enrichment (Extended Data Fig. 8G, K)

In [ ]:
import decoupler as dc

# PAGA + DPT-initialized UMAP (Extended Data Fig. 8F)
# dpt_pseudotime is pre-computed and stored in treg_strict_pseudotime.h5ad
sc.tl.paga(treg, groups="leiden_0.6")
sc.pl.paga(treg, plot=False)
sc.tl.umap(treg, init_pos="paga")

import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
umap_purple = LinearSegmentedColormap.from_list(
    "e8d4cf_a16f8d_2d223a", ["#e8d4cf", "#a16f8d", "#2d223a"]
)
ax = sc.pl.umap(
    treg, color="dpt_pseudotime", cmap=umap_purple,
    vmin=0, vmax="p99", na_color="#e8d4cf", frameon=False,
    title="Diffusion Pseudotime", show=False
)
ax.set_title("Diffusion Pseudotime")
plt.show()

In [ ]:
# TF activity via decoupler ULM (Extended Data Fig. 8G)
collectri = dc.op.collectri(organism="human")
dc.mt.ulm(data=treg, net=collectri, layer='log1p')
tf_score = dc.pp.get_obsm(treg, key="score_ulm")
treg.obsm['tf_ulm'] = pd.DataFrame(tf_score.X, index=tf_score.obs_names, columns=tf_score.var_names)

# Rank TFs by association with DPT pseudotime
tfs = dc.tl.rankby_order(tf_score, order='dpt_pseudotime')
tfs[tfs['padj'] <= 0.05][:20]

In [ ]:
top_tfs = ['KLF13','IRF5','NFKBIB','EOMES','IRF7','REL','ATF2','STAT1']

bin_tfs = dc.pp.bin_order(
    adata=tf_score,
    order="dpt_pseudotime",
    names=top_tfs,
    label="leiden_0.6",
)

fig = dc.pl.order(df=bin_tfs, mode="mat", kw_order={"vmin": -5, "vmax": +5, "cmap": "RdBu_r"}, figsize=(6, 3.2), return_fig=True)
for a, j in zip(fig.axes, range(2)):
    if j == 0:
        a.set_xlabel("Diffusion Pseudotime")

In [ ]:
# Hallmark pathway activity along DPT pseudotime (Extended Data Fig. 8G)
hallmark = dc.op.hallmark(organism="human")
dc.mt.ulm(data=treg, net=hallmark, layer='log1p')
pw_hm = dc.pp.get_obsm(treg, key="score_ulm")

hm = dc.tl.rankby_order(pw_hm, order='dpt_pseudotime')

order = [
    'Complement', 'TNF signalling via NF-kappaB', 'Inflammatory response',
    'Interferon gamma signalling', 'Hypoxia', 'EMT',
    'Fatty acid metabolism', 'TGF beta signalling', 'KRAS signalling'
]

new_names = []
name_map = {
    'COMPLEMENT': 'Complement',
    'TNFA_SIGNALING_VIA_NFKB': 'TNF signalling via NF-kappaB',
    'INFLAMMATORY_RESPONSE': 'Inflammatory response',
    'INTERFERON_GAMMA_RESPONSE': 'Interferon gamma signalling',
    'HYPOXIA': 'Hypoxia',
    'EPITHELIAL_MESENCHYMAL_TRANSITION': 'EMT',
    'FATTY_ACID_METABOLISM': 'Fatty acid metabolism',
    'KRAS_SIGNALING_UP': 'KRAS signalling',
    'TGF_BETA_SIGNALING': 'TGF beta signalling',
}
for i in hm['name'].tolist():
    new_names.append(name_map.get(i, i))
hm['new_name'] = new_names
pw_hm.var_names = new_names

exclude_hm = [
    'ALLOGRAFT_REJECTION','UV_RESPONSE_UP','PROTEIN_SECRETION','APOPTOSIS',
    'XENOBIOTIC_METABOLISM','COAGULATION','ESTROGEN_RESPONSE_EARLY',
    'KRAS_SIGNALING_DN','OXIDATIVE_PHOSPHORYLATION','ESTROGEN_RESPONSE_LATE','MYC_TARGETS_V2'
]
bin_hm = dc.pp.bin_order(
    adata=pw_hm,
    order="dpt_pseudotime",
    names=[i for i in hm['name'].tolist()[:20] if i not in exclude_hm],
    label="leiden_0.6",
)

new_names_bin = []
for i in bin_hm['name'].tolist():
    new_names_bin.append(name_map.get(i, i))
bin_hm['name'] = new_names_bin
bin_hm['name'] = pd.Categorical(bin_hm['name'], categories=order, ordered=True)

fig = dc.pl.order(df=bin_hm, mode="mat", kw_order={"vmin": -10, "vmax": +10, "cmap": "RdBu_r"}, figsize=(7, 3.5), return_fig=True)
for a, j in zip(fig.axes, range(2)):
    if j == 0:
        a.set_xlabel("Diffusion Pseudotime")

In [ ]:
# AP1 activity along DPT pseudotime (Extended Data Fig. 8K)
dc.pl.order_targets(
    adata=treg,
    net=collectri,
    label="leiden_0.6",
    source="AP1",
    order="dpt_pseudotime",
    score='tf_ulm', top=10
)

## 5. Bin pseudotime into Early / Late Treg states

In [ ]:
# Bin Tregs into 5 pseudotime quantile bins (B0–B4)
treg_mask = treg.obs['leiden_0.6'].isin(['0', '1', '2'])  # Treg clusters
pt_vals   = treg.obs.loc[treg_mask, 'palantir_pseudotime']

bins = pd.qcut(pt_vals, q=5, labels=['B0','B1','B2','B3','B4'])
treg.obs.loc[treg_mask, 'pt_bin'] = bins

# Assign Early / Late labels
treg.obs['treg_phase'] = 'Other'
treg.obs.loc[treg_mask & treg.obs['pt_bin'].isin(['B0','B1']), 'treg_phase'] = 'Early'
treg.obs.loc[treg_mask & treg.obs['pt_bin'].isin(['B2','B3','B4']), 'treg_phase'] = 'Late'

print(treg.obs['treg_phase'].value_counts())

In [ ]:
# Visualize Early vs Late on UMAP
sc.pl.umap(treg, color='treg_phase', frameon=False, s=10,
           palette={'Early':'#4393c3', 'Late':'#d6604d', 'Other':'#dddddd'})

## 5b. Treg pseudotime bin composition by condition and marker genes (Fig. 3E right, Fig. 3F)

In [ ]:
# Pseudotime bin composition by condition (Fig. 3E right panel)
pt = treg.obs["dpt_pseudotime"]
treg.obs["pt_bin"] = pd.qcut(pt, q=5, labels=False)

tbl = (
    treg.obs.groupby(["sample", "condition_summary", "pt_bin"])
    .size()
    .reset_index(name="count")
)
tbl["frac"] = tbl.groupby(["sample", "condition_summary"])["count"].transform(lambda x: x / x.sum())
tbl = tbl[tbl["frac"].notna()]
treg.obs["pt_bin"] = treg.obs["pt_bin"].astype(str)

tbl = tbl[tbl["condition_summary"] != "Control"]
tbl["condition_summary"] = tbl["condition_summary"].cat.remove_unused_categories()

plt.figure(figsize=(4, 3.7))
ax = sns.barplot(
    data=tbl,
    x="condition_summary", y="frac", hue="pt_bin",
    errorbar="se",
    estimator=np.mean,
    capsize=0.1,
    err_kws={"linewidth": 1}
)
ax.set_xlabel("Condition")
ax.set_ylabel("Fraction of cells")
ax.set_title("Treg pseudotime bin by condition")
plt.tight_layout()
plt.show()

In [ ]:
# Marker genes per pseudotime bin (Fig. 3F)
# Subset to non-Control Tregs and run rank_genes_groups on pt_bin
treg_fig3f = treg[treg.obs['condition_summary'] != 'Control'].copy()
treg_fig3f.X = treg_fig3f.layers['log1p']
sc.tl.rank_genes_groups(treg_fig3f, groupby='pt_bin', method='wilcoxon')

result = sc.get.rank_genes_groups_df(treg_fig3f, group=None)

# Read curated gene list
filtered = pd.read_excel('/share/fsmresfiles/UC/scRNA-seq/merged_cd/treg_ptbin_markers.xlsx')
plot_genes = filtered[filtered['plot'] == 'plot']['names'].unique()
result_filtered = result[result['names'].isin(plot_genes)]

In [ ]:
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from scipy.cluster.hierarchy import linkage, leaves_list

# Prepare the data
pivot_logfc = result_filtered.pivot(index='group', columns='names', values='logfoldchanges')
pivot_pvals = result_filtered.pivot(index='group', columns='names', values='pvals_adj')
group_order = ['0', '1', '2', '3', '4']
pivot_logfc = pivot_logfc.reindex(group_order)
pivot_pvals = pivot_pvals.reindex(group_order)

# Order genes by peak expression pattern
gene_peak_cluster = pivot_logfc.fillna(0).idxmax(axis=0)
gene_order_dict = {}
for gene in pivot_logfc.columns:
    peak_cluster = gene_peak_cluster[gene]
    max_expr = pivot_logfc[gene].max()
    gene_order_dict[gene] = (int(peak_cluster), -max_expr)

ordered_genes = sorted(pivot_logfc.columns, key=lambda x: gene_order_dict[x])
pivot_logfc_ordered = pivot_logfc[ordered_genes]
pivot_pvals_ordered = pivot_pvals[ordered_genes]

# Create a diverging colormap (blue-white-red)
cmap = mcolors.LinearSegmentedColormap.from_list('logfc', ['#0000FF', '#FFFFFF', '#FF0000'])

logfc_values = pivot_logfc_ordered.values.flatten()
logfc_values = logfc_values[~np.isnan(logfc_values)]
vmin, vmax = logfc_values.min(), logfc_values.max()
vabs = max(abs(vmin), abs(vmax))
norm = mcolors.Normalize(vmin=-vabs, vmax=vabs)

fig, ax = plt.subplots(figsize=(6.5, 3.2))

# Dotplot
for i, group in enumerate(pivot_logfc_ordered.index):
    for j, gene in enumerate(pivot_logfc_ordered.columns):
        logfc = pivot_logfc_ordered.loc[group, gene]
        pval = pivot_pvals_ordered.loc[group, gene]
        if pd.notna(logfc):
            size = abs(logfc) * 30
            color = cmap(norm(logfc))
            alpha = 0.9 if pval < 0.1 else 0.4
            ax.scatter(j, i, s=size, c=[color], alpha=alpha, edgecolors='black', linewidth=0.5)

ax.set_xticks(range(len(ordered_genes)))
ax.set_xticklabels(ordered_genes, rotation=90)
ax.set_yticks(range(len(group_order)))
ax.set_yticklabels(group_order)
ax.set_xlabel('Genes', fontsize=12)
ax.set_ylabel('Pseudotime Bin', fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=10, label='p_adj<0.1', alpha=0.9, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=10, label='p_adj≥0.1', alpha=0.3, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
           markersize=0, label=''),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=15, label='|Log2FC|=3', alpha=0.9, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=10, label='|Log2FC|=2', alpha=0.9, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=5, label='|Log2FC|=1', alpha=0.9, markeredgecolor='black'),
]
leg = ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left',
                title='', frameon=False, fontsize=9)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.88, 0.42, 0.125, 0.05])
cbar = plt.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.ax.tick_params(labelsize=8)
cbar.ax.set_title('Log2FC', fontsize=9, pad=5)

plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

In [ ]:
# Save pseudotime results for NicheNet analysis
treg.obs[['leiden_0.6', 'palantir_pseudotime', 'pt_bin', 'treg_phase']].to_csv(
    f"{MERGED_DIR}/treg_pseudotime_bins.csv"
)
print("Saved pseudotime bins.")